# Capital Adequacy Ratio (CAR) - Data Cleaning 

### Data Info:
* **Indicator:** `Норматив достатності (адекватності) регулятивного капіталу (не менше 10 %)` **`(CAR)`**
* **Source:** National Bank of Ukraine: https://bank.gov.ua/ua/statistic/supervision-statist
* **Raw File:** `data_raw/CAR.xlsx`
* **Output:** `data_processed/CAR_clean.xlsx`  

### Cleaning Notes:
- date row contains footnote markers such as `#`, `A4`, `⁶`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [2]:
file_path = Path("../../data_raw/Indicators_Banks.xlsx")
output_path = Path("../../data_processed/Indicators_Banks_clean.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

print(sheet_names)

['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016']


In [3]:
# inspect

df_raw = pd.read_excel(file_path, sheet_name="2025", header=None)
df_raw

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,Основні показники діяльності банків України,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(млн грн)
2,Назва показника,01.02.2025#,2025-03-01 00:00:00,01.04.2025 (А4),2025-05-01 00:00:00,2025-06-01 00:00:00,01.07.2025 (А4),2025-08-01 00:00:00,2025-09-01 00:00:00,01.10.2025 (A4),2025-11-01 00:00:00,01.12.2025#,01.01.2026#
3,1,2,3,4,5,6,7,8,9,10,11,12,13
4,Кількість діючих банків,61,61,60,60,60,60,60,60,60,60,59,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,кошти небанківських фінансових установ в націо...,56389,61049,56426,58330,60703,61410,63813,64820,64188,64127,67986,77377
86,Довідково:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87,"Рентабельність активів, %",5.76,4.94,4.64,4.66,4.59,4.63,4.61,4.62,4.59,4.53,4.55,3.58
88,"Рентабельність капіталу, %",51.1,42.78,39.78,40.13,39.89,40.17,39.86,39.61,39.05,38.28,38.18,29.98


In [4]:
# collect all sheets 

all_parts = []

indicator_rows = {
    "loans": 28, # Кредити надані клієнтам
    "capital": 53,  # Капітал
    "ROA": 87, # Рентабельність активів, %
    "ROE": 88 # Рентабельність капіталу, %
}

for sheet in sheet_names:
    
    df_raw = pd.read_excel(file_path, sheet_name=sheet, header=None)
    
    date_cells = df_raw.iloc[2, 1:]   # row 3, from column B
    
    for indicator_name, row_idx in indicator_rows.items():
        
        value_cells = df_raw.iloc[row_idx, 1:]
        
        part = pd.DataFrame({
            "date_raw": date_cells.values,
            "value": value_cells.values
        })
        
        part["indicator"] = indicator_name
        
        all_parts.append(part)

df = pd.concat(all_parts, ignore_index=True)
df

,date_raw,value,indicator
0,01.02.2025#,1136949,loans
1,2025-03-01 00:00:00,1153567,loans
2,01.04.2025 (А4),1177118,loans
3,2025-05-01 00:00:00,1197632,loans
4,2025-06-01 00:00:00,1219282,loans
...,...,...,...
479,01.09.2016#,-7.48,ROE
480,01.10.2016#,-11.54,ROE
481,01.11.2016#,-11.11,ROE
482,01.12.2016#,-15.05,ROE


In [5]:
# clean text 

df["date_raw"] = df["date_raw"].astype(str).str.strip()

date_text = (
    df["date_raw"]
    .str.replace(r"\*+", "", regex=True)            
    .str.replace(r"#\s*\d+", "", regex=True)         
    .str.replace(r"#", "", regex=True)               
    .str.replace(r"\s*\(.*?\)", "", regex=True)     
    .str.strip()
)

date_text

0               01.02.2025
1      2025-03-01 00:00:00
2               01.04.2025
3      2025-05-01 00:00:00
4      2025-06-01 00:00:00
              ...         
479             01.09.2016
480             01.10.2016
481             01.11.2016
482             01.12.2016
483             01.01.2017
Name: date_raw, Length: 484, dtype: object

In [6]:
# parse both formats and force datetime dtype

date_dmy = pd.to_datetime(
    date_text,
    format="%d.%m.%Y",
    errors="coerce"
)

date_iso = pd.to_datetime(
    date_text,
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

df["date"] = pd.to_datetime(
    date_dmy.fillna(date_iso),
    errors="coerce"
)

df["date"] = df["date"].dt.date

df["date"]

0      2025-02-01
1      2025-03-01
2      2025-04-01
3      2025-05-01
4      2025-06-01
          ...    
479    2016-09-01
480    2016-10-01
481    2016-11-01
482    2016-12-01
483    2017-01-01
Name: date, Length: 484, dtype: object

In [7]:
df_final = df[["date", "value", "indicator"]].copy()
df_final = df_final.sort_values("date").reset_index(drop=True)

df_final.tail(6)

,date,value,indicator
478,2025-12-01,481341,capital
479,2025-12-01,1380456,loans
480,2026-01-01,1220332,loans
481,2026-01-01,29.98,ROE
482,2026-01-01,463417,capital
483,2026-01-01,3.58,ROA


In [8]:
# check 

df["year"] = pd.to_datetime(df["date"], errors="coerce").dt.year

year_check = (
    df.groupby(["year", "indicator"])["value"]
    .count()
    .reset_index(name="N_values")
    .sort_values(["year", "indicator"])
)

year_check

,year,indicator,N_values
0,2016,ROA,12
1,2016,ROE,12
2,2016,capital,12
3,2016,loans,12
4,2017,ROA,12
5,2017,ROE,12
6,2017,capital,12
7,2017,loans,12
8,2018,ROA,12
9,2018,ROE,12


In [9]:
df_final.to_excel(output_path, index=False)
print(f"saved to: {output_path}")

saved to: ..\..\data_processed\Indicators_Banks_clean.xlsx
